In [ ]:
# In[1]:

import pandas as pd
import pytz
from pathlib import Path

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# helper to load and convert timestamp column if present
def load_csv_with_ts(filename):
    p = Path(filename)
    if not p.exists():
        return None
    df = pd.read_csv(p, low_memory=False)
    if 'timestamp' in df.columns and not df['timestamp'].isnull().all():
        # convert seconds -> timezone-aware datetime in Asia/Shanghai
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    return df

# Load files (reusing variables as required)
df_metric = load_csv_with_ts('metric.csv')
df_trace  = load_csv_with_ts('trace.csv')
df_log    = load_csv_with_ts('log.csv')
df_err    = load_csv_with_ts('error_logs.csv')

# Safe function to create compact overview
def compact_overview(df, ts_col='timestamp'):
    if df is None:
        return {'rows': 0, 'earliest': None, 'latest': None}
    rows = int(df.shape[0])
    if ts_col in df.columns and rows>0:
        earliest = df[ts_col].min()
        latest = df[ts_col].max()
        # format as ISO-like string without nanoseconds
        earliest_s = pd.to_datetime(earliest).isoformat()
        latest_s = pd.to_datetime(latest).isoformat()
    else:
        earliest_s = None
        latest_s = None
    return {'rows': rows, 'earliest': earliest_s, 'latest': latest_s}

# Compute overviews
metric_overview = compact_overview(df_metric)
trace_overview  = compact_overview(df_trace)
log_overview    = compact_overview(df_log)
err_overview    = compact_overview(df_err)

# Top counts helper (limit to top_n)
def top_value_counts_df(df, col, top_n=20):
    if df is None or col not in df.columns:
        return pd.DataFrame(columns=[col, 'count'])
    vc = df[col].value_counts().reset_index()
    vc.columns = [col, 'count']
    return vc.head(top_n)

# Metric specifics
metric_cmdb_top = top_value_counts_df(df_metric, 'cmdb_id', top_n=20)
metric_kpi_top  = top_value_counts_df(df_metric, 'kpi_name', top_n=20)

# Trace specifics
trace_cmdb_top  = top_value_counts_df(df_trace, 'cmdb_id', top_n=20)
trace_name_top  = top_value_counts_df(df_trace, 'trace_name', top_n=20)

# Log specifics
log_cmdb_top    = top_value_counts_df(df_log, 'cmdb_id', top_n=20)
log_name_top    = top_value_counts_df(df_log, 'log_name', top_n=20)

# Error logs specifics: top cmdb_id and up to 10 sample messages (timestamp, cmdb_id, message)
err_cmdb_top    = top_value_counts_df(df_err, 'cmdb_id', top_n=20)
if df_err is None:
    err_samples = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'message'])
else:
    # ensure timestamp and message exist
    cols = [c for c in ['timestamp','cmdb_id','message'] if c in df_err.columns]
    if len(cols) == 0:
        err_samples = pd.DataFrame(columns=['timestamp','cmdb_id','message'])
    else:
        # sort by timestamp if present, else keep original order
        if 'timestamp' in df_err.columns:
            tmp = df_err.sort_values('timestamp', ascending=True)
        else:
            tmp = df_err
        # select columns and limit to 10; convert timestamp to ISO string if present
        sel = tmp.loc[:, cols].head(10).copy()
        if 'timestamp' in sel.columns:
            sel['timestamp'] = sel['timestamp'].apply(lambda x: pd.to_datetime(x).isoformat() if pd.notnull(x) else None)
        # ensure columns order
        for c in ['timestamp','cmdb_id','message']:
            if c not in sel.columns:
                sel[c] = None
        err_samples = sel[['timestamp','cmdb_id','message']]

# Final compact JSON-like results (as variables to be displayed)
metric_overview, metric_cmdb_top, metric_kpi_top, \
trace_overview, trace_cmdb_top, trace_name_top, \
log_overview, log_cmdb_top, log_name_top, \
err_overview, err_cmdb_top, err_samples

```
Out[1]:
```
Summary of loaded telemetry (straightforward):

Overall time window
- All files cover the same ~30-minute window: earliest 2022-03-21T14:30:00+08, latest around 2022-03-21T14:59:00+08 (error_logs latest 14:59:58+08).

metric.csv
- Rows: 149,772
- Time range: 2022-03-21T14:30:00+08 — 2022-03-21T14:59:00+08
- Top cmdb_id by rows (top 20 shown): adservice (10230), adservice2 (9990), frontend2-0 (5280), frontend-0 (5160), frontend-1 (5160), frontend-2 (4980), checkoutservice-* and productcatalogservice-* also high.
- Top kpi_name (top 20 shown): dominated by container.node-5 metrics (network transmit/receive packets/MB/errors, container tasks_state, fs IO time, cpu cfs, fs_limit, fs read metrics). Many node-5 container/network and IO KPIs are prominent.

trace.csv
- Rows: 36,048
- Time range: 2022-03-21T14:30:00+08 — 2022-03-21T14:59:00+08
- Top cmdb_id by rows (top 20 shown): frontend-0 (2760), frontend-1 (2708), checkoutservice-2 (2632), checkoutservice-1 (2612), frontend-2 (2608), checkoutservice-0 (2600), etc.
- Top trace_name (top 20 shown): mainly traces from frontend pods, e.g., trace.from_frontend-0.duration_mean, duration_p95, error_rate, row_count (counts ~630), and similar patterns for other frontends and checkoutservice traces.

log.csv
- Rows: 1,814
- Time range: 2022-03-21T14:30:00+08 — 2022-03-21T14:59:00+08
- Top cmdb_id by rows: various service pods (many with 60 rows each, e.g., adservice-0/1/2, cartservice-*, checkoutservice-*, frontend-0/1/2, etc.)
- Top log_name: log.error_count (907), log.row_count (907)

error_logs.csv
- Rows: 1,540
- Time range: 2022-03-21T14:30:00+08 — 2022-03-21T14:59:58+08
- Top cmdb_id by rows (top 20 shown): frontend-0 (775), frontend-1 (461), frontend-2 (304) — error logs are concentrated on frontend pods.
- Sample messages (up to 10): all early timestamps (14:30:00 — 14:30:10+08) and show repeated warnings like:
  - severity: warning, message: failed to retrieve...
  (occurring across frontend-0, frontend-1, frontend-2)

Straightforward conclusion
- The telemetry covers a 30-minute period. Metrics are dominated by many container/node-5 network and IO KPIs. Traces are concentrated on frontend and checkoutservice spans. Logs show aggregated error_count and row_count metrics. Crucially, error_logs are heavily concentrated on frontend pods (frontend-0/1/2) with repeated "failed to retrieve..." warnings at the start of the window — this points to frontend-side problems (warnings/failures to retrieve resources) as a likely immediate symptom to investigate further.

The original code execution output of IPython Kernel is also provided below for reference:

({'rows': 149772, 'earliest': '2022-03-21T14:30:00+08:00', 'latest': '2022-03-21T14:59:00+08:00'},                      cmdb_id  count
0                  adservice  10230
1                 adservice2   9990
2                frontend2-0   5280
3                 frontend-0   5160
4                 frontend-1   5160
5                 frontend-2   4980
6          checkoutservice-0   4020
7          checkoutservice-2   4020
8          checkoutservice-1   3900
9         checkoutservice2-0   3540
10   productcatalogservice-1   3480
11   productcatalogservice-2   3480
12   productcatalogservice-0   3480
13              adservice2-0   3060
14             cartservice-1   3000
15  productcatalogservice2-0   2880
16             cartservice-0   2880
17            cartservice2-0   2880
18         shippingservice-0   2640
19             cartservice-2   2640,                                              kpi_name  count
0   container.node-5.container_network_transmit_er...   1050
1   container.node-5.container_network_transmit_pa...   1050
2   container.node-5.container_network_receive_err...   1050
3   container.node-5.container_network_transmit_pa...   1050
4   container.node-5.container_network_receive_MB....   1050
5   container.node-5.container_network_receive_pac...   1050
6   container.node-5.container_network_transmit_MB...   1050
7   container.node-5.container_network_receive_pac...   1050
8    container.node-5.container_tasks_state.iowaiting    990
9      container.node-5.container_tasks_state.stopped    990
10    container.node-5.container_tasks_state.sleeping    990
11     container.node-5.container_tasks_state.running    990
12  container.node-5.container_fs_io_time_seconds....    990
13         container.node-5.container_cpu_cfs_periods    990
14  container.node-5.container_fs_io_time_weighted...    990
15   container.node-5.container_fs_limit_MB./dev/vda1    990
16  container.node-5.container_fs_read_seconds./de...    990
17      container.node-5.container_fs_reads./dev/vda1    990
18  container.node-5.container_cpu_cfs_throttled_p...    990
19  container.node-5.container_cpu_cfs_throttled_s...    990, {'rows': 36048, 'earliest': '2022-03-21T14:30:00+08:00', 'latest': '2022-03-21T14:59:00+08:00'},                     cmdb_id  count
0                frontend-0   2760
1                frontend-1   2708
2         checkoutservice-2   2632
3         checkoutservice-1   2612
4                frontend-2   2608
5         checkoutservice-0   2600
6   productcatalogservice-1   1200
7   productcatalogservice-2   1196
8   productcatalogservice-0   1192
9               frontend2-0   1080
10       checkoutservice2-0   1080
11  recommendationservice-0    840
12  recommendationservice-2    840
13  recommendationservice-1    840
14            cartservice-2    836
15            cartservice-1    836
16            cartservice-0    836
17        currencyservice-0    720
18        currencyservice-1    720
19        currencyservice-2    720,                                     trace_name  count
0          trace.from_frontend-0.duration_mean    630
1           trace.from_frontend-0.duration_p95    630
2             trace.from_frontend-0.error_rate    630
3              trace.from_frontend-0.row_count    630
4          trace.from_frontend-1.duration_mean    617
5           trace.from_frontend-1.duration_p95    617
6             trace.from_frontend-1.error_rate    617
7              trace.from_frontend-1.row_count    617
8          trace.from_frontend-2.duration_mean    592
9           trace.from_frontend-2.duration_p95    592
10            trace.from_frontend-2.error_rate    592
11             trace.from_frontend-2.row_count    592
12      trace.from_checkoutservice-2.row_count    521
13     trace.from_checkoutservice-2.error_rate    521
14   trace.from_checkoutservice-2.duration_p95    521
15  trace.from_checkoutservice-2.duration_mean    521
16  trace.from_checkoutservice-1.duration_mean    519
17   trace.from_checkoutservice-1.duration_p95    519
18     trace.from_checkoutservice-1.error_rate    519
19      trace.from_checkoutservice-1.row_count    519, {'rows': 1814, 'earliest': '2022-03-21T14:30:00+08:00', 'latest': '2022-03-21T14:59:00+08:00'},               cmdb_id  count
0         adservice-0     60
1         adservice-1     60
2         adservice-2     60
3       cartservice-0     60
4       cartservice-1     60
5       cartservice-2     60
6   checkoutservice-0     60
7   checkoutservice-1     60
8   checkoutservice-2     60
9   currencyservice-0     60
10  currencyservice-1     60
11  currencyservice-2     60
12     emailservice-0     60
13     emailservice-1     60
14     emailservice-2     60
15         frontend-0     60
16         frontend-1     60
17         frontend-2     60
18   paymentservice-0     60
19   paymentservice-1     60,           log_name  count
0  log.error_count    907
1    log.row_count    907, {'rows': 1540, 'earliest': '2022-03-21T14:30:00+08:00', 'latest': '2022-03-21T14:59:58+08:00'},       cmdb_id  count
0  frontend-0    775
1  frontend-1    461
2  frontend-2    304,                    timestamp     cmdb_id                                            message
0  2022-03-21T14:30:00+08:00  frontend-0  severity: warning, message: failed to retrieve...
1  2022-03-21T14:30:01+08:00  frontend-1  severity: warning, message: failed to retrieve...
2  2022-03-21T14:30:01+08:00  frontend-2  severity: warning, message: failed to retrieve...
3  2022-03-21T14:30:02+08:00  frontend-0  severity: warning, message: failed to retrieve...
4  2022-03-21T14:30:04+08:00  frontend-0  severity: warning, message: failed to retrieve...
5  2022-03-21T14:30:05+08:00  frontend-1  severity: warning, message: failed to retrieve...
6  2022-03-21T14:30:06+08:00  frontend-2  severity: warning, message: failed to retrieve...
7  2022-03-21T14:30:07+08:00  frontend-0  severity: warning, message: failed to retrieve...
8  2022-03-21T14:30:09+08:00  frontend-0  severity: warning, message: failed to retrieve...
9  2022-03-21T14:30:10+08:00  frontend-1  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp is timezone-aware in Asia/Shanghai
if 'timestamp' in df_metric.columns:
    if not pd.api.types.is_datetime64tz_dtype(df_metric['timestamp']):
        df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Aggregation for specified cmdb_ids
target_cmdb = ["frontend-0","frontend-1","frontend-2",
               "checkoutservice-0","checkoutservice-1","checkoutservice-2"]

df_target = df_metric[df_metric['cmdb_id'].isin(target_cmdb)].copy()

# Define aggregation functions
def q95(x): return x.quantile(0.95)
def q05(x): return x.quantile(0.05)

grp = df_target.groupby(['cmdb_id','kpi_name'])

part1 = grp['value'].agg(
    count='count',
    p95=q95,
    p5=q05,
    mean='mean',
    std='std'
).reset_index()

# earliest and latest timestamps per group
ts_grp = df_target.groupby(['cmdb_id','kpi_name'])['timestamp'].agg(earliest='min', latest='max').reset_index()

# Merge and format timestamps as ISO strings
part1_agg = part1.merge(ts_grp, on=['cmdb_id','kpi_name'])
part1_agg['earliest'] = part1_agg['earliest'].dt.tz_convert(tz).apply(lambda x: x.isoformat())
part1_agg['latest']   = part1_agg['latest'].dt.tz_convert(tz).apply(lambda x: x.isoformat())

# Sort by count desc and limit to top 50
part1_agg = part1_agg.sort_values('count', ascending=False).reset_index(drop=True).head(50)

# 2) Top 20 distinct kpi_name containing 'node-5'
mask_node5 = df_metric['kpi_name'].str.contains('node-5', na=False)
part2_counts = df_metric.loc[mask_node5, 'kpi_name'].value_counts().reset_index()
part2_counts.columns = ['kpi_name','count']
part2_top20 = part2_counts.head(20)

# 3) For top 10 kpi_name from step 2, aggregate across all cmdb_id
top10_kpis = part2_top20['kpi_name'].head(10).tolist()
df_top10 = df_metric[df_metric['kpi_name'].isin(top10_kpis)].copy()

grp2 = df_top10.groupby('kpi_name')
part3 = grp2['value'].agg(
    count='count',
    p95=q95,
    p5=q05,
    mean='mean',
    std='std'
).reset_index()

ts_grp2 = df_top10.groupby('kpi_name')['timestamp'].agg(earliest='min', latest='max').reset_index()
part3_agg = part3.merge(ts_grp2, on='kpi_name')
part3_agg['earliest'] = part3_agg['earliest'].dt.tz_convert(tz).apply(lambda x: x.isoformat())
part3_agg['latest']   = part3_agg['latest'].dt.tz_convert(tz).apply(lambda x: x.isoformat())

# Keep only up to 10 rows (there should be <=10)
part3_agg = part3_agg.sort_values('count', ascending=False).reset_index(drop=True).head(10)

# Final compact outputs (JSON-like DataFrames)
part1_agg, part2_top20, part3_agg

```
Out[2]:
```
# Summary string of the computed aggregates (plain English)
summary = (
    "Summary of metric.csv analysis (full dataset, UTC+8):\n\n"
    "Overall time coverage:\n"
    "- Metric data span: 2022-03-21T14:30:00+08:00 through 2022-03-21T14:59:00+08:00 (30-minute window).\n\n"
    "1) Aggregates for specified components (frontend-0/1/2, checkoutservice-0/1/2):\n"
    "- Grouped by (cmdb_id, kpi_name) we computed count, P95, P5, mean, std, earliest and latest timestamps.\n"
    "- Most series in the top results have count = 30 (one value per minute across the 30-min window), and earliest/latest = 14:30 to 14:59.\n"
    "- Notable examples (from the top rows):\n"
    "  * frontend-2 / mesh.source.frontend.shippingservice.istio_response… : P95=14.275, P5=5.725, mean≈10.02, std≈2.96\n"
    "  * checkoutservice-0 / container.node-5.container_cpu_cfs_periods : P95≈26.28, P5≈7.48, mean≈16.65, std≈5.67\n"
    "  * checkoutservice-0 / container.node-5.container_cpu_cfs_throttled_… : P95≈3.55, mean≈1.42 (std≈1.12)\n"
    "\n"
    "Interpretation: the per-series counts show full 30-minute coverage. Some frontend mesh/response-time series show elevated P95 latency (~14s) relative to their P5; checkoutservice shows node-5 CPU-related KPIs present in these series.\n\n"
    "2) Top node-5 KPIs (distinct kpi_name containing 'node-5', top 20 by count):\n"
    "- Top names (examples) include: container.node-5.container_network_receive_MB..., "
    "container.node-5.container_network_receive_errors..., container.node-5.container_network_receive_packets..., "
    "container.node-5.container_network_transmit_MB..., container.node-5.container_network_transmit_errors..., "
    "container.node-5.container_network_transmit_packets..., container.node-5.container_cpu_cfs_periods, "
    "container.node-5.container_cpu_cfs_throttled_..., container.node-5.container_cpu_load_average_10s, "
    "container.node-5.container_cpu_system_seconds, container.node-5.container_cpu_usage_seconds, container.node-5.container_cpu_user_seconds, "
    "container.node-5.container_file_descriptors, container.node-5.container_memory_failures.count, fs/sector read/write metrics, etc.\n"
    "- Counts: many of these top node-5 KPIs have counts = 1050 (for network series) or 990 (for CPU/IO series), indicating wide coverage across many components/samples.\n\n"
    "3) Aggregates for top-10 node-5 KPIs (aggregated across all cmdb_id):\n"
    "- Key numeric highlights from the top-10 aggregated KPI series:\n"
    "  * container.node-5.container_network_receive_packets (count=1050): P95≈5389.2, P5≈238.7, mean≈985.39, std≈1436.50\n"
    "  * container.node-5.container_network_transmit_packets (count=1050): P95≈5402.35, P5≈163.7, mean≈960.51, std≈1523.75\n"
    "  * container.node-5.container_network_receive_MB (count=1050): P95≈0.9899, P5≈0.0216, mean≈0.1838, std≈0.2824\n"
    "  * container.node-5.container_network_transmit_MB (count=1050): P95≈4.5614, P5≈0.1303, mean≈1.1888, std≈1.3595\n"
    "  * container.node-5.container_network_receive_errors & transmit_errors (count=1050 each): P95=0, mean=0 (no errors in most samples)\n"
    "  * container.node-5.container_cpu_cfs_periods (count=990): P95≈331.5, mean≈120.19, std≈102.56\n"
    "  * container.node-5.container_cpu_cfs_throttled_* (count=990): P95 up to ~2.0, mean≈1.05 but std can be large (signaling outliers in throttling values).\n"
    "- All aggregated series share earliest/latest timestamps within the same 14:30–14:59 window.\n\n"
    "Overall concise conclusion:\n"
    "- Metric coverage is complete across the 30-minute window for the examined series.\n"
    "- Node-5 related KPIs are prominent in the dataset, especially network and CPU/IO metrics (many network packet counts show very high P95 and large variability, while MB transfer P95/means are smaller but non-zero).\n"
    "- For the selected frontend and checkoutservice series, counts are 30 (full window) and some frontend mesh traces show elevated P95 latencies; checkoutservice-related series include node-5 CPU CFS and throttling metrics. These findings suggest focusing investigation on frontend mesh latencies and node-5 network/CPU metrics (high packet counts and variability, occasional throttling outliers).\n"
    "(If you want the numeric tables or additional rows/examples, ask to display specific DataFrame slices.)"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name  count        p95       p5       mean       std                   earliest                     latest
0          frontend-2  mesh.source.frontend.shippingservice.istio_res...     30  14.275000  5.72500  10.016667  2.960904  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
1   checkoutservice-0         container.node-5.container_cpu_cfs_periods     30  26.275000  7.47500  16.650000  5.667740  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
2   checkoutservice-0  container.node-5.container_cpu_cfs_throttled_p...     30   3.550000  0.22500   1.416667  1.122523  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
3   checkoutservice-0  container.node-5.container_cpu_cfs_throttled_s...     30   0.213758  0.00166   0.074965  0.069689  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
4   checkoutservice-0    container.node-5.container_cpu_load_average_10s     30   0.000000  0.00000   0.000000  0.000000  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
..                ...                                                ...    ...        ...      ...        ...       ...                        ...                        ...
45  checkoutservice-0  mesh.destination.frontend.checkoutservice.isti...     30   0.000000  0.00000   0.000000  0.000000  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
46  checkoutservice-0  mesh.destination.frontend.checkoutservice.isti...     30   4.000000  2.50000   3.416667  0.542716  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
47  checkoutservice-0  mesh.destination.frontend.checkoutservice.isti...     30   0.000000  0.00000   0.000000  0.000000  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
48  checkoutservice-0  mesh.destination.frontend.checkoutservice.isti...     30   4.000000  2.50000   3.416667  0.542716  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
49  checkoutservice-0  mesh.destination.frontend.checkoutservice.isti...     30   0.000000  0.00000   0.000000  0.000000  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00

[50 rows x 9 columns],                                              kpi_name  count
0   container.node-5.container_network_receive_MB....   1050
1   container.node-5.container_network_receive_err...   1050
2   container.node-5.container_network_receive_pac...   1050
3   container.node-5.container_network_receive_pac...   1050
4   container.node-5.container_network_transmit_MB...   1050
5   container.node-5.container_network_transmit_er...   1050
6   container.node-5.container_network_transmit_pa...   1050
7   container.node-5.container_network_transmit_pa...   1050
8          container.node-5.container_cpu_cfs_periods    990
9   container.node-5.container_cpu_cfs_throttled_p...    990
10  container.node-5.container_cpu_cfs_throttled_s...    990
11    container.node-5.container_cpu_load_average_10s    990
12      container.node-5.container_cpu_system_seconds    990
13       container.node-5.container_cpu_usage_seconds    990
14        container.node-5.container_cpu_user_seconds    990
15        container.node-5.container_file_descriptors    990
16  container.node-5.container_memory_failures.con...    990
17  container.node-5.container_fs_reads_merged./de...    990
18  container.node-5.container_fs_sector_reads./de...    990
19  container.node-5.container_fs_sector_writes./d...    990,                                             kpi_name  count          p95          p5        mean          std                   earliest                     latest
0  container.node-5.container_network_receive_err...   1050     0.000000    0.000000    0.000000     0.000000  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
1  container.node-5.container_network_receive_MB....   1050     0.989929    0.021632    0.183816     0.282414  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
2  container.node-5.container_network_transmit_MB...   1050     4.561368    0.130267    1.188841     1.359528  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
3  container.node-5.container_network_transmit_er...   1050     0.000000    0.000000    0.000000     0.000000  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
4  container.node-5.container_network_receive_pac...   1050  5389.200000  238.725000  985.389048  1436.495143  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
5  container.node-5.container_network_receive_pac...   1050     0.000000    0.000000    0.000000     0.000000  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
6  container.node-5.container_network_transmit_pa...   1050  5402.350000  163.725000  960.514286  1523.754688  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
7  container.node-5.container_network_transmit_pa...   1050     0.000000    0.000000    0.000000     0.000000  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
8  container.node-5.container_cpu_cfs_throttled_p...    990     2.000000    0.000000    1.047980    15.434783  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
9         container.node-5.container_cpu_cfs_periods    990   331.500000    5.000000  120.187879   102.560284  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse existing df_metric if present (loaded previously)
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp column is timezone-aware Asia/Shanghai
if 'timestamp' in df_metric.columns and not pd.api.types.is_datetime64tz_dtype(df_metric['timestamp']):
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Define candidate series
target_cmdb = ["frontend-0","frontend-1","frontend-2",
               "checkoutservice-0","checkoutservice-1","checkoutservice-2"]

mask_candidates = df_metric['cmdb_id'].isin(target_cmdb) | df_metric['kpi_name'].str.contains('container.node-5', na=False)
candidates_df = df_metric.loc[mask_candidates, ['cmdb_id','kpi_name']].drop_duplicates().reset_index(drop=True)

# 2) Compute global thresholds (P95, P5) and global max/min per (cmdb_id,kpi_name) using full df_metric
grp_all = df_metric.groupby(['cmdb_id','kpi_name'])['value']
global_stats = grp_all.agg(
    global_count='count',
    global_p95=lambda x: x.quantile(0.95),
    global_p5=lambda x: x.quantile(0.05),
    global_max='max',
    global_min='min'
).reset_index()

# Keep only candidate global stats
global_stats = global_stats.merge(candidates_df, on=['cmdb_id','kpi_name'], how='inner')

# 3) Filter metric rows to incident window (inclusive)
incident_start = pd.Timestamp('2022-03-21T14:30:00', tz=tz)
incident_end   = pd.Timestamp('2022-03-21T14:59:00', tz=tz)

in_window_df = df_metric[(df_metric['timestamp'] >= incident_start) & (df_metric['timestamp'] <= incident_end)].copy()

# Keep only candidate series in-window (but we want rows even if some candidates have 0 in-window rows)
in_window_candidates = in_window_df.merge(candidates_df, on=['cmdb_id','kpi_name'], how='inner')

# Helper to compute longest consecutive run length (in minutes) for timestamps series where consecutive means +1 minute
def longest_consecutive_minutes(tseries):
    # tseries: pandas Series of timezone-aware timestamps sorted ascending
    if tseries.empty:
        return 0
    # Convert to integer minute ticks to avoid timedelta tz issues
    mins = (tseries.astype('int64') // 10**9) // 60  # minutes since epoch
    diffs = np.diff(mins)
    if diffs.size == 0:
        return 1
    # count runs where diffs==1
    runs = []
    run_len = 1
    for d in diffs:
        if d == 1:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return int(max(runs)) if runs else 0

# 4) For each candidate series compute in-window stats relative to global thresholds
records = []
# Convert global_stats to dict for fast lookup of thresholds
gs_indexed = global_stats.set_index(['cmdb_id','kpi_name'])
candidate_pairs = list(gs_indexed.index)

for cmdb_id, kpi_name in candidate_pairs:
    # global thresholds
    row_global = gs_indexed.loc[(cmdb_id,kpi_name)]
    P95 = float(row_global['global_p95'])
    P5  = float(row_global['global_p5'])
    gmax = float(row_global['global_max'])
    gmin = float(row_global['global_min'])
    # in-window rows for this series
    mask = (in_window_df['cmdb_id'] == cmdb_id) & (in_window_df['kpi_name'] == kpi_name)
    ser = in_window_df.loc[mask].sort_values('timestamp')
    count_in = int(ser.shape[0])
    # HIGH (>= P95)
    if count_in > 0:
        high_mask = ser['value'] >= P95
        count_high = int(high_mask.sum())
        if count_high > 0:
            earliest_high_ts = ser.loc[high_mask, 'timestamp'].min()
            earliest_high = earliest_high_ts.isoformat()
            max_high = float(ser['value'].max())
            # longest consecutive sequence among high points
            high_ts = ser.loc[high_mask, 'timestamp'].sort_values()
            longest_consec_high = longest_consecutive_minutes(high_ts)
        else:
            earliest_high = None
            max_high = float(ser['value'].max())  # still record max in-window
            longest_consec_high = 0
    else:
        count_high = 0
        earliest_high = None
        max_high = np.nan
        longest_consec_high = 0
    # LOW (<= P5)
    if count_in > 0:
        low_mask = ser['value'] <= P5
        count_low = int(low_mask.sum())
        if count_low > 0:
            earliest_low_ts = ser.loc[low_mask, 'timestamp'].min()
            earliest_low = earliest_low_ts.isoformat()
            min_low = float(ser['value'].min())
            low_ts = ser.loc[low_mask, 'timestamp'].sort_values()
            longest_consec_low = longest_consecutive_minutes(low_ts)
        else:
            earliest_low = None
            min_low = float(ser['value'].min())
            longest_consec_low = 0
    else:
        count_low = 0
        earliest_low = None
        min_low = np.nan
        longest_consec_low = 0
    # severity calculations
    if P95 > 0:
        severity_high = (max_high - P95) / P95 if not np.isnan(max_high) else 0.0
    else:
        severity_high = (max_high - P95) if not np.isnan(max_high) else 0.0
    if P5 > 0:
        severity_low = (P5 - min_low) / P5 if not np.isnan(min_low) else 0.0
    else:
        severity_low = (P5 - min_low) if not np.isnan(min_low) else 0.0
    # assemble record
    records.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'count_in_window': count_in,
        'count_high': count_high,
        'earliest_high': earliest_high,
        'longest_consecutive_high': int(longest_consec_high),
        'severity_high': float(severity_high),
        'count_low': count_low,
        'earliest_low': earliest_low,
        'longest_consecutive_low': int(longest_consec_low),
        'severity_low': float(severity_low),
        'global_p95': float(P95),
        'global_p5': float(P5),
        'global_max': float(gmax),
        'global_min': float(gmin)
    })

per_series_df = pd.DataFrame.from_records(records)

# 5) Return top 20 candidate series sorted by max(count_high, count_low) desc
per_series_df['max_count_side'] = per_series_df[['count_high','count_low']].max(axis=1)
result_df = per_series_df.sort_values(['max_count_side','count_in_window'], ascending=[False, False]).drop(columns=['max_count_side']).reset_index(drop=True)

# Format timestamps compactly (they are ISO strings or None already)
# Limit to top 20 rows and selected columns
display_cols = [
    'cmdb_id','kpi_name','count_in_window',
    'count_high','earliest_high','longest_consecutive_high','severity_high',
    'count_low','earliest_low','longest_consecutive_low','severity_low',
    'global_p95','global_p5','global_max','global_min'
]

result_top20 = result_df.loc[:, display_cols].head(20)

# Final displayed variable (compact)
result_top20

```
Out[3]:
```
# Plain-English summary of the computed candidate-series incident-window analysis
summary = (
    "Summary of candidate-series analysis (metric.csv, full globals, incident window 2022-03-21T14:30:00+08 through 2022-03-21T14:59:00+08):\n\n"
    "1) Candidate selection:\n"
    "- Candidates = series where cmdb_id is one of frontend-0/1/2 or checkoutservice-0/1/2, "
    "or where kpi_name contains 'container.node-5'.\n\n"
    "2) Global thresholds (computed on the entire metric.csv):\n"
    "- For each candidate we computed global P95, P5, global max and min.\n\n"
    "3) Incident-window findings (top 20 series by max(count_high,count_low)):\n"
    "- The top 20 are dominated by container.node-5 KPIs for adservice-0 (many FS/IO/CPU-related metrics).\n"
    "- For these top series: every series has 30 in-window points (full 30-minute coverage).\n"
    "- Many series have global P95 == global P5 == 0 (or P95==P5==a constant), causing both count_high and count_low to equal the full count (30)."
    " This makes severity_high and severity_low = 0 (or near-zero) because values equal the threshold.\n"
    "- Example special case: container.node-5.container_fs_limit_MB./dev/vda1 is constant ~604630.738281 (P95=P5=global_max=global_min), so again no deviation by these threshold rules.\n\n"
    "4) Interpretation and implication:\n"
    "- The threshold method flagged many series as \"high\" and \"low\" simultaneously when their global P95/P5 are degenerate (zero or constant), which reduces diagnostic value.\n"
    "- There are no clear large positive or negative severity values in the top-20 rows shown (severity_high and severity_low are zero or near-zero), i.e., no obvious extreme deviations relative to the computed global P95/P5 for these series.\n"
    "- In short: the top candidate series in this ranking appear constant across the window (either zeros or constant large values), so this particular threshold-based anomaly check did not surface strong anomalies for them.\n\n"
    "5) Recommended next steps:\n"
    "- Focus on candidate series that have non-degenerate P95/P5 (non-zero spread) or large std — e.g., network packet-count KPIs (node-5) that earlier aggregation showed very large P95 and std — for real deviations.\n"
    "- Correlate with trace/log/error_logs (frontend errors seen earlier) to prioritize series with temporal alignment to the frontend error timestamps.\n"
    "- If you want, I can (a) re-rank candidates by severity_high or severity_low instead of count, or (b) show the subset of candidates with non-zero variance to find real anomalies."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count_in_window  count_high              earliest_high  longest_consecutive_high  severity_high  count_low               earliest_low  longest_consecutive_low  severity_low     global_p95      global_p5     global_max     global_min
0   adservice-0  container.node-5.container_cpu_cfs_throttled_p...               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
1   adservice-0  container.node-5.container_cpu_cfs_throttled_s...               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
2   adservice-0    container.node-5.container_cpu_load_average_10s               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
3   adservice-0     container.node-5.container_fs_inodes./dev/vda1               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
4   adservice-0  container.node-5.container_fs_inodes_free./dev...               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
5   adservice-0  container.node-5.container_fs_io_current./dev/...               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
6   adservice-0  container.node-5.container_fs_io_time_seconds....               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
7   adservice-0  container.node-5.container_fs_io_time_weighted...               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
8   adservice-0   container.node-5.container_fs_limit_MB./dev/vda1               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0  604630.738281  604630.738281  604630.738281  604630.738281
9   adservice-0  container.node-5.container_fs_read_seconds./de...               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
10  adservice-0       container.node-5.container_fs_reads./dev/vda               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
11  adservice-0      container.node-5.container_fs_reads./dev/vda1               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
12  adservice-0    container.node-5.container_fs_reads_MB./dev/vda               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
13  adservice-0  container.node-5.container_fs_reads_merged./de...               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
14  adservice-0  container.node-5.container_fs_sector_reads./de...               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
15  adservice-0  container.node-5.container_fs_sector_writes./d...               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
16  adservice-0  container.node-5.container_fs_write_seconds./d...               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
17  adservice-0      container.node-5.container_fs_writes./dev/vda               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
18  adservice-0     container.node-5.container_fs_writes./dev/vda1               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000
19  adservice-0   container.node-5.container_fs_writes_MB./dev/vda               30          30  2022-03-21T14:30:00+08:00                        30            0.0         30  2022-03-21T14:30:00+08:00                       30           0.0       0.000000       0.000000       0.000000       0.000000```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse previously loaded dataframes if available, otherwise load
try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv', low_memory=False)
    if 'timestamp' in df_err.columns:
        df_err['timestamp'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', low_memory=False)
    if 'timestamp' in df_trace.columns:
        df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure timestamp dtype and timezone
if 'timestamp' in df_err.columns and not pd.api.types.is_datetime64tz_dtype(df_err['timestamp']):
    df_err['timestamp'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if 'timestamp' in df_trace.columns and not pd.api.types.is_datetime64tz_dtype(df_trace['timestamp']):
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) error_logs: filter frontends and aggregate per-minute
frontends = ["frontend-0","frontend-1","frontend-2"]
df_err_front = df_err[df_err['cmdb_id'].isin(frontends)].copy()

# floor to minute
df_err_front['ts_min'] = df_err_front['timestamp'].dt.floor('T')

# aggregate counts per ts_min and per frontend
err_counts = df_err_front.groupby('ts_min').size().rename('total_count').reset_index()
# counts per frontend
counts_per_front = df_err_front.pivot_table(index='ts_min', columns='cmdb_id', values='message', aggfunc='count').fillna(0).astype(int).reset_index()
# ensure columns for all three frontends
for f in frontends:
    if f not in counts_per_front.columns:
        counts_per_front[f] = 0
# merge
err_agg = err_counts.merge(counts_per_front, on='ts_min', how='left').sort_values(['total_count','ts_min'], ascending=[False, True])

# top 20 timestamps by total_count desc
err_agg_top20 = err_agg.head(20).rename(columns={'ts_min':'timestamp'})

# earliest and latest timestamp in filtered set
if not df_err_front.empty:
    err_earliest = df_err_front['ts_min'].min().isoformat()
    err_latest   = df_err_front['ts_min'].max().isoformat()
else:
    err_earliest = None
    err_latest = None
err_earliest_latest = pd.Series({'earliest': err_earliest, 'latest': err_latest})

# 2) trace.csv: filter trace_name and suffixes
trace_mask = df_trace['trace_name'].str.contains('trace.from_frontend-', na=False) & \
             df_trace['trace_name'].str.endswith(('.error_rate', '.duration_p95', '.duration_mean'))
df_trace_f = df_trace[trace_mask].copy()

# extract suffix type
def suffix_of(name):
    if name.endswith('.error_rate'):
        return 'error_rate'
    if name.endswith('.duration_p95'):
        return 'duration_p95'
    if name.endswith('.duration_mean'):
        return 'duration_mean'
    return None

df_trace_f['ts_min'] = df_trace_f['timestamp'].dt.floor('T')
df_trace_f['metric_type'] = df_trace_f['trace_name'].apply(suffix_of)

# aggregate per ts_min: mean and counts per metric_type
agg_funcs = {
    'value': ['mean','count']
}
grp = df_trace_f.groupby(['ts_min','metric_type'])['value'].agg(['mean','count']).reset_index()

# pivot so each ts_min has columns for each metric_type (mean and count)
pivot_mean = grp.pivot(index='ts_min', columns='metric_type', values='mean').rename(columns={
    'error_rate':'avg_error_rate',
    'duration_p95':'avg_duration_p95',
    'duration_mean':'avg_duration_mean'
}).reset_index()
pivot_count = grp.pivot(index='ts_min', columns='metric_type', values='count').rename(columns={
    'error_rate':'count_error_rate',
    'duration_p95':'count_duration_p95',
    'duration_mean':'count_duration_mean'
}).reset_index()

trace_agg = pivot_mean.merge(pivot_count, on='ts_min', how='left').fillna(np.nan)

# Prepare ranked lists
# a) Top 20 by avg_error_rate desc
trace_by_error = trace_agg.dropna(subset=['avg_error_rate']).copy()
trace_by_error = trace_by_error.sort_values('avg_error_rate', ascending=False).reset_index(drop=True)
trace_top_by_error = trace_by_error.loc[:, ['ts_min','avg_error_rate','count_error_rate','avg_duration_p95','avg_duration_mean']].head(20)
trace_top_by_error = trace_top_by_error.rename(columns={'ts_min':'timestamp'})

# b) Top 20 by avg_duration_p95 desc
trace_by_p95 = trace_agg.dropna(subset=['avg_duration_p95']).copy()
trace_by_p95 = trace_by_p95.sort_values('avg_duration_p95', ascending=False).reset_index(drop=True)
trace_top_by_p95 = trace_by_p95.loc[:, ['ts_min','avg_duration_p95','count_duration_p95','avg_error_rate','avg_duration_mean']].head(20)
trace_top_by_p95 = trace_top_by_p95.rename(columns={'ts_min':'timestamp'})

# Final compact outputs
err_agg_top20, err_earliest_latest, trace_top_by_error, trace_top_by_p95

```
Out[4]:
```
Summary of findings (compact, plain English):

1) error_logs (frontend-0/1/2)
- Time range in filtered frontend error logs: 2022-03-21T14:30:00+08:00 through 2022-03-21T14:59:00+08:00.
- Top per-minute error-log spikes (top 20 by total_count). Highest minutes:
  - 2022-03-21 14:49:00+08:00 — total 61 (frontend-0:31, frontend-1:17, frontend-2:13)
  - 2022-03-21 14:55:00+08:00 — total 59 (27,17,15)
  - 2022-03-21 14:47:00+08:00 — total 58 (30,14,14)
  - 2022-03-21 14:57:00+08:00 — total 58 (29,16,13)
  - 2022-03-21 14:45:00+08:00 — total 57 (31,15,11)
  - (Other high minutes include 14:43, 14:30, 14:32, 14:51, 14:35, etc.)
- Pattern: error logs are concentrated in many minute-level spikes across the 30-minute window, with frontend-0 producing the largest share in nearly every spike.

2) traces from frontends (trace.from_frontend-*, aggregated across frontends)
- We aggregated per-minute means for three trace metrics: error_rate, duration_p95, duration_mean.
- Error rates:
  - avg_error_rate is effectively 0.0 for all top-ranked minutes (no measurable aggregate error_rate across frontend traces in this window).
  - Sample sizes for error_rate are ~59–63 samples per minute (varies by minute).
- Latency (duration_p95 and duration_mean):
  - Top minutes by avg_duration_p95 (descending):
    - 2022-03-21 14:59:00+08:00 — avg_duration_p95 ≈ 0.011724 (count_duration_p95 = 60), avg_duration_mean ≈ 0.010171, avg_error_rate = 0.0
    - 2022-03-21 14:50:00+08:00 — avg_duration_p95 ≈ 0.008055 (count 63)
    - 2022-03-21 14:31:00+08:00 — avg_duration_p95 ≈ 0.007349 (count 63)
    - 2022-03-21 14:48:00+08:00 — avg_duration_p95 ≈ 0.007286 (count 63)
    - 14:42, 14:34, 14:44, ... follow with avg_duration_p95 ~0.006–0.007
  - Top minutes by avg_error_rate (descending) all show avg_error_rate = 0.0; their avg_duration_p95 values are small (0.004–0.007 range).

3) Interpretation / succinct conclusions
- Error logs show clear minute-level spikes on frontends (most prominently at ~14:49 and ~14:55), with frontend-0 contributing the majority of messages.
- Trace-level error_rate is essentially zero across the same window, meaning traces do not show aggregated “error_rate” rises even when logs spike.
- Traces do show small increases in tail latency (duration_p95) at certain minutes (peak in this dataset at 14:59 with avg_duration_p95 ≈ 0.0117, and several other minutes around 0.006–0.008).
- Combined implication: frontends are producing many log warnings/errors in discrete spikes, but distributed tracing does not record corresponding error_rate increases; there are modest latency peaks in some minutes. This suggests frontend-side failures/warnings (log noise or handled failures) without widespread trace-level errors — investigate frontend logs (message details) at the spike minutes (e.g., 14:49, 14:55, 14:47, 14:57, 14:45) and correlate with the small latency peaks (notably 14:59 and 14:50) for root-cause clues.

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp  total_count  frontend-0  frontend-1  frontend-2
19 2022-03-21 14:49:00+08:00           61          31          17          13
25 2022-03-21 14:55:00+08:00           59          27          17          15
17 2022-03-21 14:47:00+08:00           58          30          14          14
27 2022-03-21 14:57:00+08:00           58          29          16          13
15 2022-03-21 14:45:00+08:00           57          31          15          11
13 2022-03-21 14:43:00+08:00           56          29          16          11
0  2022-03-21 14:30:00+08:00           55          27          20           8
2  2022-03-21 14:32:00+08:00           55          27          20           8
21 2022-03-21 14:51:00+08:00           55          28          15          12
5  2022-03-21 14:35:00+08:00           54          26          15          13
8  2022-03-21 14:38:00+08:00           54          29          17           8
4  2022-03-21 14:34:00+08:00           53          25          18          10
23 2022-03-21 14:53:00+08:00           52          23          15          14
6  2022-03-21 14:36:00+08:00           51          26          17           8
7  2022-03-21 14:37:00+08:00           51          24          15          12
11 2022-03-21 14:41:00+08:00           51          27          14          10
29 2022-03-21 14:59:00+08:00           51          26          13          12
3  2022-03-21 14:33:00+08:00           49          24          14          11
14 2022-03-21 14:44:00+08:00           49          23          15          11
1  2022-03-21 14:31:00+08:00           48          26          10          12, earliest    2022-03-21T14:30:00+08:00
latest      2022-03-21T14:59:00+08:00
dtype: object, metric_type                 timestamp  avg_error_rate  count_error_rate  avg_duration_p95  avg_duration_mean
0           2022-03-21 14:30:00+08:00             0.0                60          0.005712           0.004979
1           2022-03-21 14:31:00+08:00             0.0                63          0.007349           0.006465
2           2022-03-21 14:32:00+08:00             0.0                59          0.004203           0.003934
3           2022-03-21 14:33:00+08:00             0.0                62          0.006882           0.006149
4           2022-03-21 14:34:00+08:00             0.0                62          0.007037           0.006500
5           2022-03-21 14:35:00+08:00             0.0                62          0.006328           0.005712
6           2022-03-21 14:36:00+08:00             0.0                62          0.005629           0.005409
7           2022-03-21 14:37:00+08:00             0.0                62          0.006102           0.005767
8           2022-03-21 14:38:00+08:00             0.0                62          0.005921           0.005631
9           2022-03-21 14:39:00+08:00             0.0                61          0.005402           0.005089
10          2022-03-21 14:40:00+08:00             0.0                62          0.005860           0.005611
11          2022-03-21 14:41:00+08:00             0.0                60          0.004875           0.004586
12          2022-03-21 14:42:00+08:00             0.0                62          0.007211           0.006665
13          2022-03-21 14:43:00+08:00             0.0                61          0.005401           0.005110
14          2022-03-21 14:44:00+08:00             0.0                62          0.007036           0.006790
15          2022-03-21 14:45:00+08:00             0.0                62          0.006561           0.006102
16          2022-03-21 14:46:00+08:00             0.0                63          0.006168           0.005831
17          2022-03-21 14:47:00+08:00             0.0                59          0.004381           0.004083
18          2022-03-21 14:48:00+08:00             0.0                63          0.007286           0.006575
19          2022-03-21 14:49:00+08:00             0.0                59          0.004322           0.004088, metric_type                 timestamp  avg_duration_p95  count_duration_p95  avg_error_rate  avg_duration_mean
0           2022-03-21 14:59:00+08:00          0.011724                  60             0.0           0.010171
1           2022-03-21 14:50:00+08:00          0.008055                  63             0.0           0.007357
2           2022-03-21 14:31:00+08:00          0.007349                  63             0.0           0.006465
3           2022-03-21 14:48:00+08:00          0.007286                  63             0.0           0.006575
4           2022-03-21 14:42:00+08:00          0.007211                  62             0.0           0.006665
5           2022-03-21 14:34:00+08:00          0.007037                  62             0.0           0.006500
6           2022-03-21 14:44:00+08:00          0.007036                  62             0.0           0.006790
7           2022-03-21 14:33:00+08:00          0.006882                  62             0.0           0.006149
8           2022-03-21 14:52:00+08:00          0.006694                  63             0.0           0.006218
9           2022-03-21 14:58:00+08:00          0.006691                  62             0.0           0.005934
10          2022-03-21 14:51:00+08:00          0.006662                  61             0.0           0.006254
11          2022-03-21 14:45:00+08:00          0.006561                  62             0.0           0.006102
12          2022-03-21 14:54:00+08:00          0.006405                  63             0.0           0.006058
13          2022-03-21 14:35:00+08:00          0.006328                  62             0.0           0.005712
14          2022-03-21 14:56:00+08:00          0.006262                  63             0.0           0.005909
15          2022-03-21 14:46:00+08:00          0.006168                  63             0.0           0.005831
16          2022-03-21 14:37:00+08:00          0.006102                  62             0.0           0.005767
17          2022-03-21 14:55:00+08:00          0.005959                  59             0.0           0.005220
18          2022-03-21 14:38:00+08:00          0.005921                  62             0.0           0.005631
19          2022-03-21 14:40:00+08:00          0.005860                  62             0.0           0.005611)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse existing df_metric if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp is timezone-aware Asia/Shanghai
if 'timestamp' in df_metric.columns and not pd.api.types.is_datetime64tz_dtype(df_metric['timestamp']):
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) spike timestamps (as timezone-aware)
spike_strs = [
  '2022-03-21T14:49:00+08:00', '2022-03-21T14:55:00+08:00', '2022-03-21T14:47:00+08:00',
  '2022-03-21T14:57:00+08:00', '2022-03-21T14:45:00+08:00', '2022-03-21T14:30:00+08:00',
  '2022-03-21T14:32:00+08:00', '2022-03-21T14:51:00+08:00', '2022-03-21T14:35:00+08:00',
  '2022-03-21T14:43:00+08:00'
]
spike_ts = [pd.Timestamp(s).tz_convert(tz) for s in spike_strs]

# helper: floor timestamps in df_metric to minute (they already should be floored)
df_metric['ts_min'] = df_metric['timestamp'].dt.floor('T')

# 2) Identify node-5 KPI names of interest
node5_mask = df_metric['kpi_name'].str.contains('container.node-5', na=False)
substrings = ['receive_packets','transmit_packets','receive_MB','transmit_MB','cpu_cfs_periods','cpu_cfs_throttled']
sub_mask = df_metric['kpi_name'].apply(lambda k: any(sub in k for sub in substrings))
node5_kpis_df = df_metric.loc[node5_mask & sub_mask, 'kpi_name'].value_counts().reset_index()
node5_kpis_df.columns = ['kpi_name','global_count']
node5_kpis_df = node5_kpis_df.head(20)  # up to 20

# store full list in kernel variable
node5_kpi_list = node5_kpis_df['kpi_name'].tolist()

# 3) For each identified node-5 kpi_name compute global P95 and P5 (group by kpi_name across full file)
node5_global_stats = df_metric[df_metric['kpi_name'].isin(node5_kpi_list)].groupby('kpi_name')['value'].agg(
    global_count='count',
    global_p95=lambda x: x.quantile(0.95),
    global_p5=lambda x: x.quantile(0.05),
    global_max='max',
    global_min='min'
).reset_index()

# 4) For each (timestamp,kpi_name) aggregate across all cmdb_id at that timestamp
rows = []
for ts in spike_ts:
    # select rows at this minute
    df_ts = df_metric[df_metric['ts_min'] == ts]
    for kpi in node5_kpi_list:
        df_k = df_ts[df_ts['kpi_name'] == kpi]
        if df_k.empty:
            agg_value = np.nan
        else:
            # choose aggregation method
            if ('packets' in kpi) or ('_MB' in kpi):
                agg_value = float(df_k['value'].sum())
            elif 'cpu_cfs' in kpi:
                agg_value = float(df_k['value'].mean())
            else:
                # fallback to sum
                agg_value = float(df_k['value'].sum())
        # get global stats
        gs = node5_global_stats[node5_global_stats['kpi_name'] == kpi].squeeze()
        if gs.empty:
            gp95 = np.nan; gp5 = np.nan
        else:
            gp95 = float(gs['global_p95'])
            gp5  = float(gs['global_p5'])
        # determine breach_type and severity
        breach = 'none'
        severity = 0.0
        if not np.isnan(agg_value):
            if (not np.isnan(gp95)) and (agg_value >= gp95):
                breach = 'high'
                severity = (agg_value - gp95) / gp95 if gp95 != 0 else (agg_value - gp95)
            elif (not np.isnan(gp5)) and (agg_value <= gp5):
                breach = 'low'
                severity = (gp5 - agg_value) / gp5 if gp5 != 0 else (gp5 - agg_value)
        rows.append({
            'timestamp': ts.isoformat(),
            'kpi_name': kpi,
            'agg_value': agg_value,
            'global_p95': gp95,
            'global_p5': gp5,
            'breach_type': breach,
            'severity': float(severity)
        })

node5_result = pd.DataFrame.from_records(rows)

# sort node5_result by severity desc then breach_type to surface breaches, then timestamp
node5_result = node5_result.sort_values(['severity','breach_type','timestamp'], ascending=[False, True, True]).reset_index(drop=True)

# 5) Frontend pods KPIs selection and processing
frontends = ['frontend-0','frontend-1','frontend-2']
frontend_mask = df_metric['cmdb_id'].isin(frontends)
frontend_kpi_mask = df_metric['kpi_name'].str.contains('cpu|network|fs_io|cfs|throttled', na=False)
frontend_candidates = df_metric[frontend_mask & frontend_kpi_mask][['cmdb_id','kpi_name']].drop_duplicates().reset_index(drop=True)

# Compute global P95/P5 per (cmdb_id,kpi_name)
frontend_global = df_metric.merge(frontend_candidates, on=['cmdb_id','kpi_name'], how='inner')
frontend_global_stats = frontend_global.groupby(['cmdb_id','kpi_name'])['value'].agg(
    global_count='count',
    global_p95=lambda x: x.quantile(0.95),
    global_p5=lambda x: x.quantile(0.05),
    global_max='max',
    global_min='min'
).reset_index()

# For each spike timestamp and each (cmdb_id,kpi_name) compute agg_value as mean across rows at that timestamp for that cmdb_id/kpi
rows_f = []
for ts in spike_ts:
    df_ts = df_metric[df_metric['ts_min'] == ts]
    for _, r in frontend_candidates.iterrows():
        cmdb = r['cmdb_id']; kpi = r['kpi_name']
        df_fk = df_ts[(df_ts['cmdb_id'] == cmdb) & (df_ts['kpi_name'] == kpi)]
        if df_fk.empty:
            agg_value = np.nan
        else:
            agg_value = float(df_fk['value'].mean())
        # global thresholds
        gs = frontend_global_stats[(frontend_global_stats['cmdb_id'] == cmdb) & (frontend_global_stats['kpi_name'] == kpi)].squeeze()
        if gs.empty:
            gp95 = np.nan; gp5 = np.nan
        else:
            gp95 = float(gs['global_p95'])
            gp5  = float(gs['global_p5'])
        breach = 'none'
        severity = 0.0
        if not np.isnan(agg_value):
            if (not np.isnan(gp95)) and (agg_value >= gp95):
                breach = 'high'
                severity = (agg_value - gp95) / gp95 if gp95 != 0 else (agg_value - gp95)
            elif (not np.isnan(gp5)) and (agg_value <= gp5):
                breach = 'low'
                severity = (gp5 - agg_value) / gp5 if gp5 != 0 else (gp5 - agg_value)
        rows_f.append({
            'timestamp': ts.isoformat(),
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'agg_value': agg_value,
            'global_p95': gp95,
            'global_p5': gp5,
            'breach_type': breach,
            'severity': float(severity)
        })

frontend_result = pd.DataFrame.from_records(rows_f)

# sort frontend_result by severity desc then timestamp
frontend_result = frontend_result.sort_values(['severity','timestamp'], ascending=[False, True]).reset_index(drop=True)

# 6) Prepare compact outputs: limit total displayed rows to <=100
# We'll show up to 50 rows from node5_result and up to 50 from frontend_result, but not more than 100 total.
n_node5_show = min(50, node5_result.shape[0])
n_front_show = min(50, frontend_result.shape[0], 100 - n_node5_show)

node5_display = node5_result.loc[:, ['timestamp','kpi_name','agg_value','global_p95','global_p5','breach_type','severity']].head(n_node5_show)
frontend_display = frontend_result.loc[:, ['timestamp','cmdb_id','kpi_name','agg_value','global_p95','global_p5','breach_type','severity']].head(n_front_show)

# Final displayed variables (compact)
node5_display, frontend_display

```
Out[5]:
```
Summary of focused correlation between error-log spike minutes and metrics (compact):

Key finding — node-5 network egress spikes
- At several error-log spike minutes (examples: 14:43, 14:35, 14:49, 14:57, 14:55) the aggregated node-5 network transmit (MB) across hosts is much higher than normal:
  - Example: container.node-5.container_network_transmit_MB aggregated ≈ 43 MB at 14:43 vs global P95 ≈ 4.56 MB → breach_type = high, severity ≈ 8.6 (i.e., ~8.6× above P95).
  - Similar large transmit_MB spikes occur at 14:35, 14:49, 14:57, 14:55 (agg ≈ 42–43 MB; severity ≈ 8.4–8.6).
- These large transmit_MB spikes align with several of the top frontend error-log spike minutes, indicating heavy outgoing network traffic on node-5 during error bursts.

Frontend pod-level observations
- Some frontend pods show elevated per-pod network receive/transmit metrics at spike minutes:
  - Example: frontend-1 at 14:55 — container_network_receive_MB ≈ 0.841 vs global P95 ≈ 0.319 → breach high, severity ≈ 1.63.
  - frontend-1 at 14:35 — container_network_transmit_packets aggregated ≈ 2,531 vs global P95 ≈ 2,382 → breach high (smaller relative severity ≈ 0.06).
- Some frontend CPU-related metrics are low at specific minutes (e.g., frontend-1 container_cpu_system_seconds at 14:47 flagged low relative to P5), suggesting reduced system CPU time at that minute for that pod — worth checking but lower magnitude than the network signals.

Caveats
- Several KPI series have degenerate global thresholds (P95 or P5 == 0 or constant), causing many rows to appear as 'high' by the <=/>= checks even when severity is 0. Focus on non-degenerate metrics with non-zero severity for meaningful signals.
- The strongest, non-degenerate signal here is node-5 network transmit MB (large absolute and relative spikes).

Concise conclusion and recommended next steps
- There is a clear correlation between frontend error-log spikes and large node-5 network transmit spikes during many of the same minutes. This points to a network egress/throughput event on node-5 around those spike times as a likely contributor or symptom.
- Next actions:
  1) Inspect node-5 network interface metrics and host-level logs around the minutes (14:35, 14:43, 14:49, 14:55, 14:57, etc.) for packet drops, retransmits, or saturation.
  2) Correlate with frontend pod logs at those exact minutes (messages already show warnings) and check whether outgoing traffic corresponds to retries or large responses.
  3) Investigate routing/load-balancer behavior and any bursty traffic sources at those minutes.
  4) Re-run focused checks on packet loss/retransmit KPIs and per-pod network latency to confirm causality.

The original code execution output of IPython Kernel is also provided below for reference:

(                    timestamp                                           kpi_name  agg_value  global_p95  global_p5 breach_type  severity
0   2022-03-21T14:43:00+08:00  container.node-5.container_network_transmit_MB...  43.667408    4.561368   0.130267        high  8.573314
1   2022-03-21T14:35:00+08:00  container.node-5.container_network_transmit_MB...  43.264904    4.561368   0.130267        high  8.485072
2   2022-03-21T14:49:00+08:00  container.node-5.container_network_transmit_MB...  43.099174    4.561368   0.130267        high  8.448739
3   2022-03-21T14:57:00+08:00  container.node-5.container_network_transmit_MB...  43.022928    4.561368   0.130267        high  8.432023
4   2022-03-21T14:55:00+08:00  container.node-5.container_network_transmit_MB...  42.740285    4.561368   0.130267        high  8.370059
..                        ...                                                ...        ...         ...        ...         ...       ...
45  2022-03-21T14:35:00+08:00  container.node-5.container_network_transmit_pa...   0.000000    0.000000   0.000000        high  0.000000
46  2022-03-21T14:43:00+08:00  container.node-5.container_network_receive_pac...   0.000000    0.000000   0.000000        high  0.000000
47  2022-03-21T14:43:00+08:00  container.node-5.container_network_transmit_pa...   0.000000    0.000000   0.000000        high  0.000000
48  2022-03-21T14:45:00+08:00  container.node-5.container_network_receive_pac...   0.000000    0.000000   0.000000        high  0.000000
49  2022-03-21T14:45:00+08:00  container.node-5.container_network_transmit_pa...   0.000000    0.000000   0.000000        high  0.000000

[50 rows x 7 columns],                     timestamp     cmdb_id                                           kpi_name      agg_value     global_p95      global_p5 breach_type  severity
0   2022-03-21T14:55:00+08:00  frontend-1  container.node-5.container_network_receive_MB....       0.840842       0.319423       0.239729        high  1.632381
1   2022-03-21T14:55:00+08:00  frontend-2  container.node-5.container_network_receive_MB....       0.475991       0.361570       0.165569        high  0.316458
2   2022-03-21T14:47:00+08:00  frontend-1      container.node-5.container_cpu_system_seconds       0.100000       0.188250       0.106750         low  0.063232
3   2022-03-21T14:35:00+08:00  frontend-1  container.node-5.container_network_transmit_pa...    2531.000000    2382.700000    1822.975000        high  0.062240
4   2022-03-21T14:55:00+08:00  frontend-1  container.node-5.container_network_receive_pac...    1836.500000    1739.525000    1345.400000        high  0.055748
..                        ...         ...                                                ...            ...            ...            ...         ...       ...
45  2022-03-21T14:30:00+08:00  frontend-0  container.node-5.container_network_transmit_pa...       0.000000       0.000000       0.000000        high  0.000000
46  2022-03-21T14:30:00+08:00  frontend-0         container.node-5.container_spec_cpu_period  100000.000000  100000.000000  100000.000000        high  0.000000
47  2022-03-21T14:30:00+08:00  frontend-0          container.node-5.container_spec_cpu_quota   40000.000000   40000.000000   40000.000000        high  0.000000
48  2022-03-21T14:30:00+08:00  frontend-0         container.node-5.container_spec_cpu_shares     204.000000     204.000000     204.000000        high  0.000000
49  2022-03-21T14:30:00+08:00  frontend-1         container.node-5.container_cpu_cfs_periods     139.500000     191.375000     139.500000         low  0.000000

[50 rows x 8 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[6]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse loaded dataframes if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv', low_memory=False)
    if 'timestamp' in df_err.columns:
        df_err['timestamp'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', low_memory=False)
    if 'timestamp' in df_trace.columns:
        df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure timestamps are timezone-aware Asia/Shanghai
if 'timestamp' in df_metric.columns and not pd.api.types.is_datetime64tz_dtype(df_metric['timestamp']):
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if 'timestamp' in df_err.columns and not pd.api.types.is_datetime64tz_dtype(df_err['timestamp']):
    df_err['timestamp'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if 'timestamp' in df_trace.columns and not pd.api.types.is_datetime64tz_dtype(df_trace['timestamp']):
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Floor to minute columns
df_metric['ts_min'] = df_metric['timestamp'].dt.floor('T')
df_err['ts_min'] = df_err['timestamp'].dt.floor('T')
df_trace['ts_min'] = df_trace['timestamp'].dt.floor('T')

# Candidate timestamps (as ISO strings)
spike_ts_strs = ['2022-03-21T14:30:00+08:00','2022-03-21T14:49:00+08:00','2022-03-21T14:50:00+08:00']
spike_ts = [pd.Timestamp(s).tz_convert(tz) for s in spike_ts_strs]

# 1) Node-5 network transmit MB check
kpi_tx_pattern = 'container.node-5.container_network_transmit_MB'
# Full-file aggregated time series: sum across cmdb_id per minute for transmit MB
mask_tx = df_metric['kpi_name'].str.contains(kpi_tx_pattern, na=False)
agg_tx_series = df_metric.loc[mask_tx].groupby('ts_min')['value'].sum().reset_index().rename(columns={'value':'agg_transmit_MB'})

# Compute global P95 and P5 on the aggregated series (full-file)
global_p95 = float(agg_tx_series['agg_transmit_MB'].quantile(0.95)) if not agg_tx_series['agg_transmit_MB'].empty else np.nan
global_p5  = float(agg_tx_series['agg_transmit_MB'].quantile(0.05)) if not agg_tx_series['agg_transmit_MB'].empty else np.nan

# For each spike timestamp get aggregated value and compute breach/severity
rows_tx = []
for ts in spike_ts:
    row = agg_tx_series[agg_tx_series['ts_min'] == ts]
    agg_val = float(row['agg_transmit_MB'].iloc[0]) if not row.empty else np.nan
    breach = 'none'
    severity = 0.0
    if not np.isnan(agg_val):
        if not np.isnan(global_p95) and agg_val >= global_p95:
            breach = 'high'
            severity = (agg_val - global_p95)/global_p95 if global_p95 != 0 else (agg_val - global_p95)
        elif not np.isnan(global_p5) and agg_val <= global_p5:
            breach = 'low'
            severity = (global_p5 - agg_val)/global_p5 if global_p5 != 0 else (global_p5 - agg_val)
    rows_tx.append({
        'timestamp': ts.isoformat(),
        'agg_transmit_MB': agg_val,
        'global_p95': global_p95,
        'global_p5': global_p5,
        'breach_type': breach,
        'severity': float(severity)
    })
node5_transmit_table = pd.DataFrame.from_records(rows_tx)

# 2) Frontend error_log counts per timestamp
frontends = ['frontend-0','frontend-1','frontend-2']
err_counts = []
for ts in spike_ts:
    df_ts = df_err[df_err['ts_min'] == ts]
    total = int(df_ts.shape[0])
    c0 = int(df_ts[df_ts['cmdb_id']=='frontend-0'].shape[0])
    c1 = int(df_ts[df_ts['cmdb_id']=='frontend-1'].shape[0])
    c2 = int(df_ts[df_ts['cmdb_id']=='frontend-2'].shape[0])
    err_counts.append({
        'timestamp': ts.isoformat(),
        'total_error_count': total,
        'count_frontend-0': c0,
        'count_frontend-1': c1,
        'count_frontend-2': c2
    })
error_counts_table = pd.DataFrame.from_records(err_counts)

# 3) Frontend trace summary per timestamp
# Filter frontend traces
mask_front_traces = df_trace['trace_name'].str.contains('trace.from_frontend-', na=False)
df_trace_f = df_trace[mask_front_traces].copy()

rows_tr = []
for ts in spike_ts:
    df_ts = df_trace_f[df_trace_f['ts_min'] == ts]
    # error_rate
    er = df_ts[df_ts['trace_name'].str.endswith('.error_rate', na=False)]['value']
    dp95 = df_ts[df_ts['trace_name'].str.endswith('.duration_p95', na=False)]['value']
    dmean = df_ts[df_ts['trace_name'].str.endswith('.duration_mean', na=False)]['value']
    avg_error_rate = float(er.mean()) if not er.empty else np.nan
    avg_duration_p95 = float(dp95.mean()) if not dp95.empty else np.nan
    avg_duration_mean = float(dmean.mean()) if not dmean.empty else np.nan
    sample_count = int(df_ts.shape[0])
    rows_tr.append({
        'timestamp': ts.isoformat(),
        'avg_error_rate': avg_error_rate,
        'avg_duration_p95': avg_duration_p95,
        'avg_duration_mean': avg_duration_mean,
        'number_of_frontend_trace_samples': sample_count
    })
trace_summary_table = pd.DataFrame.from_records(rows_tr)

# Display the three compact tables (exactly 3 rows each)
node5_transmit_table, error_counts_table, trace_summary_table

```
Out[7]:
```
Compact confirmation for the three timestamps (UTC+8):

Node-5 transmit MB (global P95=43.7825, P5=38.9080)
- 2022-03-21T14:30:00+08:00 — agg_transmit_MB = 40.4347 — between P5 and P95 => breach_type: none
- 2022-03-21T14:49:00+08:00 — agg_transmit_MB = 43.0992 — slightly below P95 => breach_type: none
- 2022-03-21T14:50:00+08:00 — agg_transmit_MB = 42.3848 — below P95 => breach_type: none

Frontend error-log counts (per-minute)
- 2022-03-21T14:30:00+08:00 — total = 55 (frontend-0:27, frontend-1:20, frontend-2:8)
- 2022-03-21T14:49:00+08:00 — total = 61 (frontend-0:31, frontend-1:17, frontend-2:13)
- 2022-03-21T14:50:00+08:00 — total = 45 (frontend-0:24, frontend-1:14, frontend-2:7)

Frontend trace summary (aggregated across frontend traces)
- 2022-03-21T14:30:00+08:00 — avg_error_rate = 0.0, avg_duration_p95 ≈ 0.00571, avg_duration_mean ≈ 0.00498, samples = 240
- 2022-03-21T14:49:00+08:00 — avg_error_rate = 0.0, avg_duration_p95 ≈ 0.00432, avg_duration_mean ≈ 0.00409, samples = 236
- 2022-03-21T14:50:00+08:00 — avg_error_rate = 0.0, avg_duration_p95 ≈ 0.00806, avg_duration_mean ≈ 0.00736, samples = 252

Concise interpretation
- All three minutes show high node-5 transmit MB values (≈40–43 MB) but none exceed the global P95 threshold — they are high but just below the 95th-percentile cutoff.
- Error-log spikes are clear and strongest at 14:49 (total 61), with frontend-0 contributing the largest share in each minute.
- Traces show no increase in aggregated error_rate (avg_error_rate = 0.0) but small latency bumps; notably 14:50 has the largest avg_duration_p95 among the three (~0.008s).
- Taken together: frontend error-log spikes align with elevated (near-tail) node-5 transmit MB and modest frontend tail-latency increases, especially around 14:49–14:50. This supports network egress/bandwidth activity on node-5 as a plausible contributor; next steps: inspect node-5 packet-level metrics (retransmits, drops), host network stats, and frontend logs around 14:49–14:50.

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp  agg_transmit_MB  global_p95  global_p5 breach_type  severity
0  2022-03-21T14:30:00+08:00        40.434742   43.782478  38.908022        none       0.0
1  2022-03-21T14:49:00+08:00        43.099174   43.782478  38.908022        none       0.0
2  2022-03-21T14:50:00+08:00        42.384758   43.782478  38.908022        none       0.0,                    timestamp  total_error_count  count_frontend-0  count_frontend-1  count_frontend-2
0  2022-03-21T14:30:00+08:00                 55                27                20                 8
1  2022-03-21T14:49:00+08:00                 61                31                17                13
2  2022-03-21T14:50:00+08:00                 45                24                14                 7,                    timestamp  avg_error_rate  avg_duration_p95  avg_duration_mean  number_of_frontend_trace_samples
0  2022-03-21T14:30:00+08:00             0.0          0.005712           0.004979                               240
1  2022-03-21T14:49:00+08:00             0.0          0.004322           0.004088                               236
2  2022-03-21T14:50:00+08:00             0.0          0.008055           0.007357                               252)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# reuse existing dataframes from kernel if present
tz = pytz.timezone('Asia/Shanghai')
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if 'ts_min' not in df_metric.columns:
    df_metric['ts_min'] = df_metric['timestamp'].dt.floor('T')
# ensure trace and error logs available
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', low_memory=False)
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    df_trace['ts_min'] = df_trace['timestamp'].dt.floor('T')
if 'ts_min' not in df_trace.columns:
    df_trace['ts_min'] = df_trace['timestamp'].dt.floor('T')

try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv', low_memory=False)
    df_err['timestamp'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    df_err['ts_min'] = df_err['timestamp'].dt.floor('T')
if 'ts_min' not in df_err.columns:
    df_err['ts_min'] = df_err['timestamp'].dt.floor('T')

# 1) Full global aggregated series of node-5 total transmit MB (sum across cmdb_id per minute)
kpi_tx_pattern = 'container.node-5.container_network_transmit_MB'
mask_tx = df_metric['kpi_name'].str.contains(kpi_tx_pattern, na=False)
agg_tx_series_full = df_metric.loc[mask_tx].groupby('ts_min', as_index=False)['value'].sum().rename(columns={'value':'agg_transmit_MB'})

# Compute global P95 and P90 on the aggregated series (full-file)
global_p95 = float(agg_tx_series_full['agg_transmit_MB'].quantile(0.95)) if not agg_tx_series_full.empty else np.nan
global_p90 = float(agg_tx_series_full['agg_transmit_MB'].quantile(0.90)) if not agg_tx_series_full.empty else np.nan

# 2) Filter to incident window and select timestamps where agg_transmit_MB >= P90
incident_start = pd.Timestamp('2022-03-21T14:30:00', tz=tz)
incident_end   = pd.Timestamp('2022-03-21T14:59:00', tz=tz)

agg_tx_incident = agg_tx_series_full[
    (agg_tx_series_full['ts_min'] >= incident_start) & (agg_tx_series_full['ts_min'] <= incident_end)
].copy()

# select >= P90
candidates = agg_tx_incident[agg_tx_incident['agg_transmit_MB'] >= global_p90].copy()
# add breach_level and severity_vs_p95
def compute_breach(row):
    v = row['agg_transmit_MB']
    if not np.isnan(global_p95) and v >= global_p95:
        level = '>=P95'
    else:
        level = '>=P90'
    if not np.isnan(global_p95) and global_p95 != 0:
        sev = (v - global_p95) / global_p95
    elif not np.isnan(global_p95):
        sev = v - global_p95
    else:
        sev = 0.0
    return pd.Series([level, float(sev)])
candidates[['breach_level','severity_vs_p95']] = candidates.apply(compute_breach, axis=1)
candidates = candidates.sort_values('agg_transmit_MB', ascending=False).head(10).reset_index(drop=True)
# format timestamp column
candidates['timestamp'] = candidates['ts_min'].dt.tz_convert(tz).apply(lambda x: x.isoformat())

node5_candidates_table = candidates.loc[:, ['timestamp','agg_transmit_MB']].assign(
    global_p95=global_p95, global_p90=global_p90
)
node5_candidates_table['breach_level'] = candidates['breach_level']
node5_candidates_table['severity_vs_p95'] = candidates['severity_vs_p95']

# 3) For selected timestamps get correlated signals: error_logs counts and frontend trace aggregates
selected_ts = candidates['ts_min'].tolist()

# Prepare error log aggregates per selected timestamp
frontends = ['frontend-0','frontend-1','frontend-2']
err_agg = []
for ts in selected_ts:
    df_e = df_err[df_err['ts_min'] == ts]
    total = int(df_e.shape[0])
    c0 = int(df_e[df_e['cmdb_id']=='frontend-0'].shape[0])
    c1 = int(df_e[df_e['cmdb_id']=='frontend-1'].shape[0])
    c2 = int(df_e[df_e['cmdb_id']=='frontend-2'].shape[0])
    err_agg.append({'ts_min':ts, 'total_error_count':total, 'count_frontend-0':c0, 'count_frontend-1':c1, 'count_frontend-2':c2})
err_agg_df = pd.DataFrame(err_agg)

# Prepare trace aggregates per selected timestamp (frontend traces)
mask_front_traces = df_trace['trace_name'].str.contains('trace.from_frontend-', na=False)
df_trace_f = df_trace[mask_front_traces].copy()

trace_rows = []
for ts in selected_ts:
    df_t = df_trace_f[df_trace_f['ts_min'] == ts]
    er = df_t[df_t['trace_name'].str.endswith('.error_rate', na=False)]['value']
    dp95 = df_t[df_t['trace_name'].str.endswith('.duration_p95', na=False)]['value']
    dmean = df_t[df_t['trace_name'].str.endswith('.duration_mean', na=False)]['value']
    avg_error_rate = float(er.mean()) if not er.empty else np.nan
    avg_duration_p95 = float(dp95.mean()) if not dp95.empty else np.nan
    avg_duration_mean = float(dmean.mean()) if not dmean.empty else np.nan
    sample_count = int(df_t.shape[0])
    trace_rows.append({'ts_min':ts, 'avg_error_rate':avg_error_rate, 'avg_duration_p95':avg_duration_p95, 'avg_duration_mean':avg_duration_mean, 'sample_count':sample_count})
trace_agg_df = pd.DataFrame(trace_rows)

# Merge node5 candidates with error logs and trace aggregates
merged = candidates.merge(err_agg_df, left_on='ts_min', right_on='ts_min', how='left').merge(trace_agg_df, on='ts_min', how='left')
# format output row per timestamp combining fields
combined_table = merged[['ts_min','agg_transmit_MB','total_error_count','count_frontend-0','count_frontend-1','count_frontend-2','avg_error_rate','avg_duration_p95','avg_duration_mean','sample_count']].copy()
combined_table['timestamp'] = combined_table['ts_min'].dt.tz_convert(tz).apply(lambda x: x.isoformat())
combined_table = combined_table[['timestamp','agg_transmit_MB','total_error_count','count_frontend-0','count_frontend-1','count_frontend-2','avg_error_rate','avg_duration_p95','avg_duration_mean','sample_count']]

# Limit to at most 10 rows (already limited)
node5_candidates_table, combined_table

```
Out[8]:
```
Summary of candidate failure minutes (compact):

Key metrics (node-5 transmit P90=43.5170, P95=43.7825):
- 2022-03-21T14:39:00+08:00
  - node-5 agg_transmit_MB = 46.3337 → >=P95 (severity vs P95 ≈ 0.058)
  - error_logs: total=48 (frontend-0:23, frontend-1:13, frontend-2:12)
  - trace: avg_error_rate=0.0, avg_duration_p95≈0.00540s, avg_duration_mean≈0.00509s, samples=244

- 2022-03-21T14:31:00+08:00
  - node-5 agg_transmit_MB = 43.8766 → >=P95 (severity ≈ 0.002)
  - error_logs: total=48 (frontend-0:26, frontend-1:10, frontend-2:12)
  - trace: avg_error_rate=0.0, avg_duration_p95≈0.00735s, avg_duration_mean≈0.00647s, samples=252

- 2022-03-21T14:43:00+08:00
  - node-5 agg_transmit_MB = 43.6674 → >=P90 but just below P95 (severity vs P95 ≈ -0.0026)
  - error_logs: total=56 (frontend-0:29, frontend-1:16, frontend-2:11)
  - trace: avg_error_rate=0.0, avg_duration_p95≈0.00540s, avg_duration_mean≈0.00511s, samples=244

Interpretation and next steps (concise)
- Highest node-5 egress spike is at 14:39 (only minute clearly above P95). 14:31 also slightly above P95; 14:43 is just above P90.
- Error-log volume is elevated across these minutes (largest at 14:43), but trace error_rate remains 0.0 and latency increases are modest.
- Likely correlation: bursts of node-5 outbound traffic coincide with frontend error-log spikes, especially 14:39 and 14:43. These minutes are prime candidates for deeper inspection.
- Recommend: inspect node-5 network packet-level KPIs (retransmits/drops/packets), host NIC stats and routing, and frontend logs at those exact minutes (14:39 and 14:43 first) to confirm root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp  agg_transmit_MB  global_p95  global_p90 breach_level  severity_vs_p95
0  2022-03-21T14:39:00+08:00        46.333651   43.782478   43.517013        >=P95         0.058269
1  2022-03-21T14:31:00+08:00        43.876626   43.782478   43.517013        >=P95         0.002150
2  2022-03-21T14:43:00+08:00        43.667408   43.782478   43.517013        >=P90        -0.002628,                    timestamp  agg_transmit_MB  total_error_count  count_frontend-0  count_frontend-1  count_frontend-2  avg_error_rate  avg_duration_p95  avg_duration_mean  sample_count
0  2022-03-21T14:39:00+08:00        46.333651                 48                23                13                12             0.0          0.005402           0.005089           244
1  2022-03-21T14:31:00+08:00        43.876626                 48                26                10                12             0.0          0.007349           0.006465           252
2  2022-03-21T14:43:00+08:00        43.667408                 56                29                16                11             0.0          0.005401           0.005110           244)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
from pathlib import Path

tz = pytz.timezone('Asia/Shanghai')

# reuse df_metric if present, else load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)
    if 'timestamp' in df_metric.columns:
        df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure kpi_name exists
if 'kpi_name' not in df_metric.columns:
    node5_packet_kpis = pd.DataFrame(columns=['kpi_name','count'])
else:
    # mask: contains 'container.node-5' AND any of the substrings (case-sensitive)
    substr_regex = r'packet|packets|error|errors|retrans|drop'
    mask = df_metric['kpi_name'].str.contains('container.node-5', na=False) & df_metric['kpi_name'].str.contains(substr_regex, na=False)
    node5_packet_kpis = df_metric.loc[mask, ['kpi_name']].groupby('kpi_name', as_index=False).size().rename(columns={'size':'count'})
    node5_packet_kpis = node5_packet_kpis.sort_values('count', ascending=False).reset_index(drop=True).head(50)

# Display compact result (up to 50 rows)
node5_packet_kpis

```
Out[9]:
```
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)
    if 'timestamp' in df_metric.columns:
        df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Find node-5 KPIs containing packet/error/retrans/drop substrings (case-sensitive)
substr_regex = r'packet|packets|error|errors|retrans|drop'
mask = df_metric['kpi_name'].str.contains('container.node-5', na=False) & df_metric['kpi_name'].str.contains(substr_regex, na=False)
node5_packet_kpis = df_metric.loc[mask, ['kpi_name']].groupby('kpi_name', as_index=False).size().rename(columns={'size':'count'})
node5_packet_kpis = node5_packet_kpis.sort_values('count', ascending=False).reset_index(drop=True).head(50)

# Build concise summary string
lines = []
for _, r in node5_packet_kpis.iterrows():
    lines.append(f"{r['kpi_name']} — {int(r['count'])}")
summary = "Node-5 packet/error-related KPIs (kpi_name — count):\n" + "\n".join(lines)

summary

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  count
0  container.node-5.container_network_receive_err...   1050
1  container.node-5.container_network_receive_pac...   1050
2  container.node-5.container_network_receive_pac...   1050
3  container.node-5.container_network_transmit_er...   1050
4  container.node-5.container_network_transmit_pa...   1050
5  container.node-5.container_network_transmit_pa...   1050```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# ensure ts_min exists and is tz-aware Asia/Shanghai
if 'ts_min' not in df_metric.columns:
    df_metric['ts_min'] = df_metric['timestamp'].dt.floor('T')
else:
    if not pd.api.types.is_datetime64tz_dtype(df_metric['ts_min']):
        df_metric['ts_min'] = pd.to_datetime(df_metric['ts_min']).dt.tz_localize(tz)

# 1) Identify node-5 packet/error KPIs
substr_regex = r'packet|packets|error|errors|retrans|drop'
mask = df_metric['kpi_name'].str.contains('container.node-5', na=False) & df_metric['kpi_name'].str.contains(substr_regex, na=False)
node5_kpis = df_metric.loc[mask, 'kpi_name'].unique().tolist()

# Compute global stats per kpi_name over full series (no time filtering)
global_stats = df_metric[df_metric['kpi_name'].isin(node5_kpis)].groupby('kpi_name')['value'].agg(
    global_count='count',
    global_P95=lambda x: float(x.quantile(0.95)),
    global_P5=lambda x: float(x.quantile(0.05)),
    global_max='max',
    global_min='min'
).reset_index()

# 2) Candidate timestamps (UTC+8)
cand_strs = [
  '2022-03-21T14:31:00+08:00',
  '2022-03-21T14:39:00+08:00',
  '2022-03-21T14:43:00+08:00',
  '2022-03-21T14:49:00+08:00',
  '2022-03-21T14:55:00+08:00',
  '2022-03-21T14:30:00+08:00'
]
cand_ts = [pd.Timestamp(s).tz_convert(tz).floor('T') for s in cand_strs]

# Aggregate SUM across all cmdb_id at each candidate timestamp for each kpi_name
rows = []
for ts in cand_ts:
    for kpi in node5_kpis:
        df_k_ts = df_metric[(df_metric['kpi_name'] == kpi) & (df_metric['ts_min'] == ts)]
        agg_val = float(df_k_ts['value'].sum()) if not df_k_ts.empty else np.nan
        gs = global_stats[global_stats['kpi_name'] == kpi]
        if gs.empty:
            gp95 = np.nan; gp5 = np.nan
        else:
            gp95 = float(gs['global_P95'].iloc[0]); gp5 = float(gs['global_P5'].iloc[0])
        breach = 'none'; severity = 0.0
        if not np.isnan(agg_val):
            if (not np.isnan(gp95)) and (agg_val >= gp95):
                breach = 'high'
                severity = (agg_val - gp95) / gp95 if gp95 != 0 else (agg_val - gp95)
            elif (not np.isnan(gp5)) and (agg_val <= gp5):
                breach = 'low'
                severity = (gp5 - agg_val) / gp5 if gp5 != 0 else (gp5 - agg_val)
        rows.append({
            'timestamp': ts.isoformat(),
            'kpi_name': kpi,
            'agg_value': agg_val,
            'global_P95': gp95,
            'global_P5': gp5,
            'breach_type': breach,
            'severity': float(severity)
        })

result_df = pd.DataFrame.from_records(rows)

# 3) Sort by timestamp then severity desc, limit to 50 rows
# Convert timestamp column to datetime-aware for sorting without tz_localize error
ts_parsed = pd.to_datetime(result_df['timestamp'])
# ensure tz-aware: if already tz-aware this is fine; convert to tz
if pd.api.types.is_datetime64tz_dtype(ts_parsed):
    ts_parsed = ts_parsed.dt.tz_convert(tz)
else:
    ts_parsed = ts_parsed.dt.tz_localize(tz)
result_df['timestamp_dt'] = ts_parsed
result_df = result_df.sort_values(['timestamp_dt','severity'], ascending=[True, False]).drop(columns=['timestamp_dt']).reset_index(drop=True).head(50)

# Display compact result
result_df.loc[:, ['timestamp','kpi_name','agg_value','global_P95','global_P5','breach_type','severity']]

```
Out[10]:
```
Summary (compact, plain English):

What I checked
- For node-5 packet/error KPIs I computed global P95/P5 over the full file, then summed each KPI across all cmdb_id at the candidate minutes and measured breaches.

Key findings
- Strong packet-volume spikes at the candidate minutes. Example highlights:
  - 2022-03-21T14:30:00+08:00
    - container_network_receive_packets (node-5, sum) = 34,440 vs global P95 ≈ 5,389 → breach high, severity ≈ 5.39×
    - container_network_transmit_packets (node-5, sum) = 33,518 vs global P95 ≈ 5,402 → breach high, severity ≈ 5.20×
  - Similar extreme transmit/receive packet sums (many tens of thousands) appear at other candidate minutes (14:31, 14:39, 14:43, 14:49, 14:55) with severity values ≈ 4–5× the P95 for packet KPIs.
- Error-count KPIs (receive_err / transmit_err) are 0 at these minutes; because their global P95 is 0, they appear flagged as “high” by the simple >=P95 rule but with zero severity — i.e., no observed packet errors despite the large packet volumes.

Interpretation
- The dominant anomaly is volume: node-5 experienced very large bursts of packet transmit/receive at these minutes (well above normal 95th-percentile levels). These are clear network throughput/traffic spikes rather than immediate packet-error spikes.
- Frontend error-log spikes we saw earlier align temporally with these high packet-volume minutes — consistent with heavy outbound/inbound traffic correlating with frontend warnings/errors.

Recommended next checks
1. Inspect node-5 packet-level health: retransmits, drops, NIC utilization, queue lengths, interface errors and retransmit counters around the high-volume minutes.
2. Correlate with frontend logs at the exact minutes to see whether retries/large responses or upstream traffic flows caused the bursts.
3. Check load-balancer or external traffic sources for sudden bursts at the same times.
4. Treat packet-volume saturation separately from packet errors: high traffic can cause timeouts or application-level errors even if raw error counters remain zero.

If you want, I can produce a concise table of the top N timestamps with numeric severity values and the corresponding packet sums to help you pick the final two failure minutes.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp                                           kpi_name  agg_value  global_P95  global_P5 breach_type  severity
0   2022-03-21T14:30:00+08:00  container.node-5.container_network_receive_pac...    34440.0     5389.20    238.725        high  5.390559
1   2022-03-21T14:30:00+08:00  container.node-5.container_network_transmit_pa...    33518.5     5402.35    163.725        high  5.204430
2   2022-03-21T14:30:00+08:00  container.node-5.container_network_receive_err...        0.0        0.00      0.000        high  0.000000
3   2022-03-21T14:30:00+08:00  container.node-5.container_network_receive_pac...        0.0        0.00      0.000        high  0.000000
4   2022-03-21T14:30:00+08:00  container.node-5.container_network_transmit_er...        0.0        0.00      0.000        high  0.000000
..                        ...                                                ...        ...         ...        ...         ...       ...
31  2022-03-21T14:55:00+08:00  container.node-5.container_network_transmit_pa...    34459.0     5402.35    163.725        high  5.378520
32  2022-03-21T14:55:00+08:00  container.node-5.container_network_receive_err...        0.0        0.00      0.000        high  0.000000
33  2022-03-21T14:55:00+08:00  container.node-5.container_network_receive_pac...        0.0        0.00      0.000        high  0.000000
34  2022-03-21T14:55:00+08:00  container.node-5.container_network_transmit_er...        0.0        0.00      0.000        high  0.000000
35  2022-03-21T14:55:00+08:00  container.node-5.container_network_transmit_pa...        0.0        0.00      0.000        high  0.000000

[36 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing dataframes if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv', low_memory=False)
    df_err['timestamp'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', low_memory=False)
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure ts_min exists (minute-floor) and tz-aware
if 'ts_min' not in df_metric.columns:
    df_metric['ts_min'] = df_metric['timestamp'].dt.floor('T')
else:
    df_metric['ts_min'] = pd.to_datetime(df_metric['ts_min']).dt.tz_localize(tz) if not pd.api.types.is_datetime64tz_dtype(df_metric['ts_min']) else df_metric['ts_min']

if 'ts_min' not in df_err.columns:
    df_err['ts_min'] = df_err['timestamp'].dt.floor('T')
else:
    df_err['ts_min'] = pd.to_datetime(df_err['ts_min']).dt.tz_localize(tz) if not pd.api.types.is_datetime64tz_dtype(df_err['ts_min']) else df_err['ts_min']

if 'ts_min' not in df_trace.columns:
    df_trace['ts_min'] = df_trace['timestamp'].dt.floor('T')
else:
    df_trace['ts_min'] = pd.to_datetime(df_trace['ts_min']).dt.tz_localize(tz) if not pd.api.types.is_datetime64tz_dtype(df_trace['ts_min']) else df_trace['ts_min']

# 1) Build full-file aggregated per-minute series
# node_tx_mb_ts: exact match
mask_tx_mb = df_metric['kpi_name'] == 'container.node-5.container_network_transmit_MB'
node_tx_mb_ts = df_metric.loc[mask_tx_mb].groupby('ts_min', as_index=False)['value'].sum().rename(columns={'value':'agg_tx_mb'})

# node_tx_packets_ts: kpi_name contains transmit packet(s)
mask_tx_packets = df_metric['kpi_name'].str.contains('container_network_transmit_packet', na=False)
node_tx_packets_ts = df_metric.loc[mask_tx_packets].groupby('ts_min', as_index=False)['value'].sum().rename(columns={'value':'agg_tx_packets'})

# node_rx_packets_ts: kpi_name contains receive packet(s)
mask_rx_packets = df_metric['kpi_name'].str.contains('container_network_receive_packet', na=False)
node_rx_packets_ts = df_metric.loc[mask_rx_packets].groupby('ts_min', as_index=False)['value'].sum().rename(columns={'value':'agg_rx_packets'})

# Merge full series into one DataFrame for aligned timestamps (outer join)
all_ts = node_tx_mb_ts.merge(node_tx_packets_ts, on='ts_min', how='outer').merge(node_rx_packets_ts, on='ts_min', how='outer')
all_ts = all_ts.fillna(0)

# 2) Compute global P95 for each aggregated series (full-file)
p95_tx_mb = float(all_ts['agg_tx_mb'].quantile(0.95)) if not all_ts['agg_tx_mb'].empty else np.nan
p95_tx_packets = float(all_ts['agg_tx_packets'].quantile(0.95)) if not all_ts['agg_tx_packets'].empty else np.nan
p95_rx_packets = float(all_ts['agg_rx_packets'].quantile(0.95)) if not all_ts['agg_rx_packets'].empty else np.nan

# 3) Restrict to incident window and compute composite severity
start = pd.Timestamp('2022-03-21T14:30:00', tz=tz)
end   = pd.Timestamp('2022-03-21T14:59:00', tz=tz)
window_ts = all_ts[(all_ts['ts_min'] >= start) & (all_ts['ts_min'] <= end)].copy()

# safe ratio function: if p95>0 -> agg/p95, if p95==0 and agg>0 -> inf, if both zero -> 0
def safe_ratio(agg, p95):
    if np.isnan(p95) or np.isnan(agg):
        return np.nan
    if p95 > 0:
        return agg / p95
    else:
        if agg > 0:
            return np.inf
        else:
            return 0.0

window_ts['ratio_tx_mb'] = window_ts['agg_tx_mb'].apply(lambda v: safe_ratio(v, p95_tx_mb))
window_ts['ratio_tx_packets'] = window_ts['agg_tx_packets'].apply(lambda v: safe_ratio(v, p95_tx_packets))
window_ts['ratio_rx_packets'] = window_ts['agg_rx_packets'].apply(lambda v: safe_ratio(v, p95_rx_packets))

window_ts['composite_severity'] = window_ts[['ratio_tx_mb','ratio_tx_packets','ratio_rx_packets']].max(axis=1)

# 4) Select top 5 timestamps by composite_severity desc
top5 = window_ts.sort_values('composite_severity', ascending=False).head(5).reset_index(drop=True)

# 5) For each selected timestamp gather error_logs counts and frontend trace aggregates
frontends = ['frontend-0','frontend-1','frontend-2']
err_group = df_err.groupby(['ts_min','cmdb_id']).size().unstack(fill_value=0)

# prepare trace filtered for frontend traces
df_trace_front = df_trace[df_trace['trace_name'].str.contains('trace.from_frontend-', na=False)].copy()

trace_aggs = []
for ts in top5['ts_min']:
    # error logs
    total_err = int(df_err[df_err['ts_min'] == ts].shape[0])
    c0 = int(err_group.loc[ts]['frontend-0']) if (ts in err_group.index and 'frontend-0' in err_group.columns) else int(df_err[(df_err['ts_min']==ts)&(df_err['cmdb_id']=='frontend-0')].shape[0])
    c1 = int(err_group.loc[ts]['frontend-1']) if (ts in err_group.index and 'frontend-1' in err_group.columns) else int(df_err[(df_err['ts_min']==ts)&(df_err['cmdb_id']=='frontend-1')].shape[0])
    c2 = int(err_group.loc[ts]['frontend-2']) if (ts in err_group.index and 'frontend-2' in err_group.columns) else int(df_err[(df_err['ts_min']==ts)&(df_err['cmdb_id']=='frontend-2')].shape[0])
    # trace aggregates for this minute
    df_t = df_trace_front[df_trace_front['ts_min'] == ts]
    er = df_t[df_t['trace_name'].str.endswith('.error_rate', na=False)]['value']
    dp95 = df_t[df_t['trace_name'].str.endswith('.duration_p95', na=False)]['value']
    dmean = df_t[df_t['trace_name'].str.endswith('.duration_mean', na=False)]['value']
    avg_er = float(er.mean()) if not er.empty else np.nan
    avg_dp95 = float(dp95.mean()) if not dp95.empty else np.nan
    avg_dmean = float(dmean.mean()) if not dmean.empty else np.nan
    sample_count = int(df_t.shape[0])
    trace_aggs.append({'ts_min': ts, 'total_error_count': total_err, 'count_frontend-0': c0, 'count_frontend-1': c1, 'count_frontend-2': c2,
                       'frontend_trace_avg_error_rate': avg_er, 'frontend_trace_avg_duration_p95': avg_dp95, 'frontend_trace_avg_duration_mean': avg_dmean,
                       'frontend_trace_sample_count': sample_count})

trace_agg_df = pd.DataFrame(trace_aggs)

# Merge top5 with trace/error aggregates and format output
out = top5.merge(trace_agg_df, left_on='ts_min', right_on='ts_min', how='left')

# Final compact columns as requested
out_display = out[['ts_min',
                   'agg_tx_mb', 
                   ]]
# Add required columns stepwise to ensure exact order
out_display['p95_tx_mb'] = p95_tx_mb
out_display['agg_tx_packets'] = out['agg_tx_packets']
out_display['p95_tx_packets'] = p95_tx_packets
out_display['agg_rx_packets'] = out['agg_rx_packets']
out_display['p95_rx_packets'] = p95_rx_packets
out_display['composite_severity'] = out['composite_severity']
out_display['error_log_total'] = out['total_error_count']
out_display['error_log_frontend-0'] = out['count_frontend-0']
out_display['error_log_frontend-1'] = out['count_frontend-1']
out_display['error_log_frontend-2'] = out['count_frontend-2']
out_display['frontend_trace_avg_error_rate'] = out['frontend_trace_avg_error_rate']
out_display['frontend_trace_avg_duration_p95'] = out['frontend_trace_avg_duration_p95']
out_display['frontend_trace_avg_duration_mean'] = out['frontend_trace_avg_duration_mean']

# Format timestamp to ISO strings for display and rename column to 'timestamp'
out_display = out_display.rename(columns={'ts_min':'timestamp'})
out_display['timestamp'] = out_display['timestamp'].dt.tz_convert(tz).apply(lambda x: x.isoformat())

# Keep exactly the specified columns in order
final_cols = ['timestamp',
              'agg_tx_mb','p95_tx_mb',
              'agg_tx_packets','p95_tx_packets',
              'agg_rx_packets','p95_rx_packets',
              'composite_severity',
              'error_log_total','error_log_frontend-0','error_log_frontend-1','error_log_frontend-2',
              'frontend_trace_avg_error_rate','frontend_trace_avg_duration_p95','frontend_trace_avg_duration_mean']

top5_df = out_display.loc[:, final_cols].sort_values('composite_severity', ascending=False).reset_index(drop=True)

# Display the top-5 compact table
top5_df

```
Out[11]:
```
Summary (compact, plain English):

What was ranked
- I ranked minutes in the incident window by a composite severity = max(agg_tx_mb/p95_tx_mb, agg_tx_packets/p95_tx_packets, agg_rx_packets/p95_rx_packets). Because the MB series is zero/absent (p95_tx_mb=0), the ranking is driven by packet-count signals.

Top 5 candidate failure minutes (ranked)
1) 2022-03-21T14:39:00+08:00
   - agg_tx_packets = 51,024.3 (p95_tx_packets ≈ 49,576.8)
   - agg_rx_packets = 50,377.0 (p95_rx_packets ≈ 49,459.7)
   - composite_severity ≈ 1.029 (≈ 2.9% above p95)
   - error logs: total=48 (frontend-0:23, frontend-1:13, frontend-2:12)
   - frontend traces: avg_error_rate = 0.0, avg_duration_p95 ≈ 0.005402 s, avg_duration_mean ≈ 0.005089 s

2) 2022-03-21T14:56:00+08:00
   - agg_tx_packets = 49,836.3 (≈ p95 49,576.8)
   - agg_rx_packets = 49,632.2 (≈ p95 49,459.7)
   - composite_severity ≈ 1.005
   - error logs: total=44 (frontend-0:24, frontend-1:14, frontend-2:6)
   - frontend traces: avg_duration_p95 ≈ 0.006262 s

3) 2022-03-21T14:58:00+08:00
   - agg_tx_packets = 49,259.7 (slightly below p95)
   - agg_rx_packets = 49,249.0 (slightly below p95)
   - composite_severity ≈ 0.996
   - error logs: total=48 (frontend-0:26, frontend-1:14, frontend-2:8)
   - frontend traces: avg_duration_p95 ≈ 0.006691 s

4) 2022-03-21T14:30:00+08:00
   - agg_tx_packets = 48,540.0
   - agg_rx_packets = 48,385.8
   - composite_severity ≈ 0.979
   - error logs: total=55 (frontend-0:27, frontend-1:20, frontend-2:8)
   - frontend traces: avg_duration_p95 ≈ 0.005712 s

5) 2022-03-21T14:55:00+08:00
   - agg_tx_packets = 48,534.2
   - agg_rx_packets = 48,132.3
   - composite_severity ≈ 0.979
   - error logs: total=59 (frontend-0:27, frontend-1:17, frontend-2:15)
   - frontend traces: avg_duration_p95 ≈ 0.005959 s

Interpretation and next steps (concise)
- Packet-count bursts on node-5 (tx and rx) are the strongest signal aligning with frontend error-log spikes. 14:39 is the top candidate (clearly >P95), followed by minutes near the upper tail (14:56, 14:58, 14:30, 14:55).
- Traces show avg_error_rate = 0.0 and only modest latency increases (duration_p95 values on the order of milliseconds), so the issue appears as heavy packet-volume bursts rather than trace-level errors.
- Recommended immediate checks for the top minutes (start with 14:39, then 14:56 and 14:58):
  1) Node-5 network internals: retransmits/drops, NIC utilization, queue saturation, interrupt stats.
  2) Per-pod network packet metrics and frontend logs at the exact minute to see if large responses or retries caused the bursts.
  3) Upstream/ingress sources (load-balancer or external traffic) that could explain sudden packet surges.

If you want, I can now produce per-pod packet and retransmit/detail metrics for these top minutes to pinpoint the root-cause pod or process.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp  agg_tx_mb  p95_tx_mb  agg_tx_packets  p95_tx_packets  agg_rx_packets  p95_rx_packets  composite_severity  error_log_total  error_log_frontend-0  error_log_frontend-1  error_log_frontend-2  frontend_trace_avg_error_rate  frontend_trace_avg_duration_p95  frontend_trace_avg_duration_mean
0  2022-03-21T14:39:00+08:00        0.0        0.0    51024.333333    49576.833333    50377.000000    49459.741667            1.029197               48                    23                    13                    12                            0.0                         0.005402                          0.005089
1  2022-03-21T14:56:00+08:00        0.0        0.0    49836.333333    49576.833333    49632.166667    49459.741667            1.005234               44                    24                    14                     6                            0.0                         0.006262                          0.005909
2  2022-03-21T14:58:00+08:00        0.0        0.0    49259.666667    49576.833333    49249.000000    49459.741667            0.995739               48                    26                    14                     8                            0.0                         0.006691                          0.005934
3  2022-03-21T14:30:00+08:00        0.0        0.0    48540.000000    49576.833333    48385.833333    49459.741667            0.979086               55                    27                    20                     8                            0.0                         0.005712                          0.004979
4  2022-03-21T14:55:00+08:00        0.0        0.0    48534.166667    49576.833333    48132.333333    49459.741667            0.978969               59                    27                    17                    15                            0.0                         0.005959                          0.005220```
```